# VideoRAG PD 워크스테이션 — Gradio 통합 데모 (v2)

Scene Graph(JSON) 입력 → 3경로 분기(USE_AS_IS / TRANSFORM / GENERATE) → PD 실시간 결정 → 영상 합성

## 주요 기능
- **PD 인터랙티브 워크플로**: 장면별로 프롬프트 수정, 백엔드 선택, 승인/재시도/건너뛰기/파일업로드
- **단순 검색**: 텍스트 쿼리 → 영상 검색 + 합성 (한국어 Papago 자동 번역)
- **PD 큐레이션**: search_only → 클립 선택/순서 변경 → assemble_curated


In [ ]:
# ── Step 0: 환경 설정 ──
# 00_setup.ipynb, 01_indexing.ipynb를 이미 실행한 상태에서 사용하세요.
# (런타임 재연결 시에도 이 셀만 다시 실행하면 됩니다)

!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive 마운트 완료")

import sys, os, shutil, importlib, json

# ── 경로 CONFIG (00_setup과 동일) ──
PROJECT_ROOT = '/content/videorag_prototype'
DRIVE_PROJECT = '/content/drive/MyDrive/videorag_prototype'
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data', 'msrvtt')
VIDEO_DIR    = os.path.join(DATA_DIR, 'videos')
KEYFRAME_DIR = os.path.join(DATA_DIR, 'keyframes')
INDEX_DIR    = os.path.join(PROJECT_ROOT, 'index')
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, 'output')
DRIVE_INDEX  = os.path.join(DRIVE_PROJECT, 'index')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 의존성 설치 (검색/합성에 필요한 패키지) ──
!pip install -q faiss-cpu rank_bm25 transformers timm einops \
    moviepy opencv-python cryptography scikit-learn scipy \
    tqdm runwayml matplotlib pandas numpy gradio langdetect requests openai 2>/dev/null
!pip install -q ragatouille colbert-ai 2>/dev/null
!pip install -q open_clip_torch huggingface_hub>=0.19.0 easydict 2>/dev/null
print('✓ 의존성 설치 완료')

# ── Papago API 키 등록 (Colab Secrets → 환경변수) ──
from google.colab import userdata
try:
    os.environ["PAPAGO_CLIENT_ID"] = userdata.get("PAPAGO_CLIENT_ID")
    os.environ["PAPAGO_CLIENT_SECRET"] = userdata.get("PAPAGO_CLIENT_SECRET")
    print('✓ Papago API 키 설정 완료')
except Exception as e:
    print(f'⚠ Papago API 키 미등록 — 한국어 쿼리 번역이 작동하지 않습니다: {e}')

# ── HF_TOKEN 설정 (InternVideo2 모델 다운로드용) ──
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print('✓ HF_TOKEN 설정 완료')
except Exception:
    print('⚠ HF_TOKEN 미등록 — InternVideo2 다운로드 시 인증 실패 가능')

# ── sys.path 등록 ──
sys.path.insert(0, PROJECT_ROOT)

# ── GitHub에서 최신 src 동기화 ──
# [Public repo] token 불필요 — public clone
os.system("rm -rf /tmp/VideoRAG-Prototype")
os.system("git clone https://github.com/LimPark996/VideoRAG-Public.git /tmp/VideoRAG-Prototype")
os.system(f"rm -rf {PROJECT_ROOT}/src")
os.system(f"cp -r /tmp/VideoRAG-Prototype/src {PROJECT_ROOT}/src")
print('✓ src 동기화 완료')

# ── InternVideo2 설치 (sparse checkout) ──
INTERNVIDEO_PATH = '/content/InternVideo/InternVideo2/multi_modality'
if not os.path.exists(os.path.join(INTERNVIDEO_PATH, 'demo')):
    os.chdir('/content')
    if os.path.exists('/content/InternVideo'):
        shutil.rmtree('/content/InternVideo')
    !git clone --no-checkout --depth=1 https://github.com/OpenGVLab/InternVideo.git /content/InternVideo
    %cd /content/InternVideo
    !git sparse-checkout init --cone
    !git sparse-checkout set InternVideo2/multi_modality
    !git checkout main
    os.chdir('/content')
    print('✓ InternVideo2 sparse checkout 완료')
else:
    print('✓ InternVideo2 이미 존재')

if INTERNVIDEO_PATH not in sys.path:
    sys.path.insert(0, INTERNVIDEO_PATH)
importlib.invalidate_caches()

# ── 인덱스 복원 (Drive → 로컬) ──
if not os.path.exists(INDEX_DIR) or not os.listdir(INDEX_DIR):
    if os.path.exists(DRIVE_INDEX) and os.listdir(DRIVE_INDEX):
        os.makedirs(INDEX_DIR, exist_ok=True)
        for f in os.listdir(DRIVE_INDEX):
            shutil.copy2(os.path.join(DRIVE_INDEX, f), os.path.join(INDEX_DIR, f))
        print(f'✓ Drive에서 인덱스 복원: {os.listdir(INDEX_DIR)}')
    else:
        print('⚠ 인덱스 없음 — 01_indexing.ipynb를 먼저 실행하세요')
else:
    print(f'✓ 인덱스 존재: {os.listdir(INDEX_DIR)}')

# ── 영상 복원 (1k-A 테스트셋 기준 — 01_indexing과 동일 로직) ──
# 인덱스는 msrvtt_test_1k.json 대상 영상으로 구축됨 (test set, video6513~)
# train_val ZIP이 아닌 1k-A annotation 대상 영상을 복원해야 함
import zipfile, urllib.request

ANNOTATION_JSON = os.path.join(DATA_DIR, "annotations", "msrvtt_test_1k.json")
DRIVE_VIDEOS    = os.path.join(DRIVE_PROJECT, "data", "msrvtt", "videos")
HF_ZIP_URL      = "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip"
HF_ZIP_PATH     = "/content/MSRVTT_Videos.zip"

os.makedirs(VIDEO_DIR, exist_ok=True)

# 1k-A annotation 다운로드 (없으면 HuggingFace에서)
if not os.path.exists(ANNOTATION_JSON):
    _ann_url = "https://huggingface.co/datasets/friedrichor/MSR-VTT/raw/main/msrvtt_test_1k.json"
    os.makedirs(os.path.dirname(ANNOTATION_JSON), exist_ok=True)
    print(f"1k-A annotation 다운로드 중...")
    urllib.request.urlretrieve(_ann_url, ANNOTATION_JSON)
    print(f"✓ 다운로드 완료: {ANNOTATION_JSON}")
else:
    print(f"✓ Annotation 이미 존재: {ANNOTATION_JSON}")

# 1k-A 대상 video_id 목록 로드
with open(ANNOTATION_JSON, "r") as _f:
    _ann = json.load(_f)
_test_ids = {s["video_id"] for s in _ann}
print(f"✓ 1k-A test video ID: {len(_test_ids)}개")

# (a) Drive에서 복원
if os.path.exists(DRIVE_VIDEOS):
    _copied, _skipped = 0, 0
    for vid in _test_ids:
        src = os.path.join(DRIVE_VIDEOS, f"{vid}.mp4")
        dst = os.path.join(VIDEO_DIR, f"{vid}.mp4")
        if not os.path.exists(dst):
            if os.path.exists(src):
                try:
                    os.symlink(src, dst)
                except OSError:
                    shutil.copy2(src, dst)
                _copied += 1
        else:
            _skipped += 1
    print(f"✓ Drive 연결: {_copied}개 신규, {_skipped}개 이미 존재")

# (b) 여전히 부족하면 HuggingFace zip에서 추출
_have = {f[:-4] for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")} & _test_ids
_missing = sorted(_test_ids - _have)
if _missing:
    print(f"⚠ {len(_missing)}개 부족 → HuggingFace에서 다운로드 중... (2.19 GB)")
    if not os.path.exists(HF_ZIP_PATH):
        urllib.request.urlretrieve(HF_ZIP_URL, HF_ZIP_PATH)
        print(f"✓ 다운로드 완료")
    _missing_set = set(_missing)
    _extracted = 0
    with zipfile.ZipFile(HF_ZIP_PATH) as _zf:
        for member in _zf.namelist():
            name = os.path.basename(member)
            if not name.endswith(".mp4") or name[:-4] not in _missing_set:
                continue
            dst = os.path.join(VIDEO_DIR, name)
            if os.path.exists(dst) and os.path.getsize(dst) > 0:
                continue
            with _zf.open(member) as _src, open(dst, "wb") as _out:
                shutil.copyfileobj(_src, _out)
            _extracted += 1
    print(f"✓ zip 추출 완료: {_extracted}개")

# 최종 검증
_have_final = {f[:-4] for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")} & _test_ids
_still_missing = sorted(_test_ids - _have_final)
print(f"✓ 대상 영상 수: {len(_have_final)}개 / {len(_test_ids)}개 (1k-A test set)")
if _still_missing:
    print(f"  ⚠ 누락 영상 {len(_still_missing)}개: {_still_missing[:5]}{'...' if len(_still_missing)>5 else ''}")
else:
    print("  → 영상 복원 완료")


print(f"\n{'='*60}")
print(f"02_demo: 환경 설정 완료")
print(f"  VIDEO_DIR:  {VIDEO_DIR}")
print(f"  INDEX_DIR:  {INDEX_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")
print(f"{'='*60}")


In [ ]:
# ── Step 1: 파이프라인 + StoryboardMapper 초기화 ──
import time, json, os
import numpy as np

os.makedirs(OUTPUT_DIR, exist_ok=True)

from src.pipeline import VideoRAGPipeline
from src.data_models import CurationState, ClipResult
from src.phase4_assembly.storyboard_mapper import (
    StoryboardMapper, SceneRequirement, MappedScene,
    PDReviewRequest, PDExecutionResult, BranchDecision,
)
from src.phase4_assembly.inverse_prompt_engine import InversePromptEngine

from google.colab import userdata
try:
    os.environ["RUNWAY_API_KEY"] = userdata.get("RUNWAY_API_KEY")
    print('✓ Video Generation API 키 설정 완료')
except Exception as e:
    print(f"  Runway API: '미설정 (OpenCV 폴백)'{e}")

# ── Pipeline ──
config = {
    "embed_dim": 512,
    "w_visual": 0.6,
    "w_text": 0.4,
    "rrf_k": 60,
    "papago_client_id": os.environ.get("PAPAGO_CLIENT_ID", ""),
    "papago_client_secret": os.environ.get("PAPAGO_CLIENT_SECRET", ""),
    "video_dir": DRIVE_VIDEOS,  # Drive 영상 경로 — 로컬 없을 때 fallback
}

pipeline = VideoRAGPipeline(
    index_dir=INDEX_DIR,
    output_dir=OUTPUT_DIR,
    config=config,
)
pipeline.load_index()

# ITM 활성화 상태 확인 (itm_vision_features.pt 존재 시 자동 활성)
if pipeline.itm_scorer is not None:
    print("✓ ITMScorer 활성화 — Phase 3b ITM 재순위 사용 (Dense→ColBERT→ITM)")
else:
    print("ℹ  ITMScorer 비활성 — Phase 3a ColBERT까지만 적용")
    print("   활성화하려면: 01_indexing.ipynb Step 3.5 실행 후 load_index() 재호출")

# ── StoryboardMapper ──
inverse_engine = InversePromptEngine(
    openai_api_key=os.environ.get("OPENAI_API_KEY", ""),
    runway_api_key=os.environ.get("RUNWAY_API_KEY", ""),
)

# ── Runway 진단 ──
if inverse_engine.runway_api_key:
    print(f"  Runway API 키: 설정됨 ({inverse_engine.runway_api_key[:8]}...)")
    try:
        import runwayml
        print(f"  runwayml 패키지: v{runwayml.__version__} 설치됨")
    except ImportError:
        print("  ⚠️ runwayml 패키지 미설치 → pip install runwayml")
else:
    print("  ⚠️ RUNWAY_API_KEY 미설정 → Runway 백엔드 사용 불가 (OpenCV만 가능)")
    print("    설정: Colab Secrets에 RUNWAY_API_KEY 추가 또는 os.environ['RUNWAY_API_KEY'] = '...'") 


def search_fn(query, top_k=10):
    curation = pipeline.search_only(query, top_k=top_k)
    return curation.search_results

mapper = StoryboardMapper(
    search_fn=search_fn,
    inverse_engine=inverse_engine,
    top_k=10,
)

print("=" * 60)
print("Step 1: 초기화 완료")
print("=" * 60)
# ── 모델 웜업 (첫 쿼리 지연 제거) ──
print("모델 웜업 중...")
import time as _t
_w0 = _t.perf_counter()
pipeline.load_index()

# ITM 활성화 상태 확인 (itm_vision_features.pt 존재 시 자동 활성)
if pipeline.itm_scorer is not None:
    print("✓ ITMScorer 활성화 — Phase 3b ITM 재순위 사용 (Dense→ColBERT→ITM)")
else:
    print("ℹ  ITMScorer 비활성 — Phase 3a ColBERT까지만 적용")
    print("   활성화하려면: 01_indexing.ipynb Step 3.5 실행 후 load_index() 재호출")
_ = pipeline.embedder.encode_query("warmup query")
print(f"  InternVideo2 임베딩 웜업 완료: {(_t.perf_counter()-_w0)*1000:.0f}ms")
_w1 = _t.perf_counter()
try:
    pipeline.reranker._load_model()
except Exception:
    pass
print(f"  ColBERT 웜업 완료: {(_t.perf_counter()-_w1)*1000:.0f}ms")
print("모델 웜업 완료 — 이후 검색은 모델 로딩 없이 실행됩니다.")


In [ ]:
# ── Step 2: Gradio PD 워크스테이션 (v4) ──
# Tab 1: Scene Graph 워크플로 (자동 분기 + PD 리뷰 + 장면 재배치)
# Tab 2: PD 큐레이션 (수동 검색 + 클립 선택 + 변환/생성 + 합성)
import gradio as gr
import json, os, time, traceback, threading, queue, cv2
import numpy as np

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import subprocess as _sp


# ═════════════════════════════════════════════════════════
# Gradio 로그 패널 — 모든 에러/경고를 UI에 표시
# ═════════════════════════════════════════════════════════
import logging as _logging

class _GradioLogCapture(_logging.Handler):
    """Python logging → Gradio 로그 패널 전달용 핸들러"""
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        ts = time.strftime("%H:%M:%S")
        lvl = record.levelname
        msg = self.format(record)
        self.records.append(f"[{ts}] {lvl} | {msg}")
    def flush_text(self):
        return "\n".join(self.records[-200:])  # 최근 200줄
    def clear(self):
        self.records.clear()

_log_capture = _GradioLogCapture()
_log_capture.setLevel(_logging.INFO)
# root logger 하나에만 핸들러 등록 (자식 로거는 propagate로 자동 전달)
_logging.getLogger().setLevel(_logging.INFO)
_logging.getLogger().addHandler(_log_capture)

def _get_logs():
    """현재까지 쌓인 로그 텍스트 반환"""
    return _log_capture.flush_text() or "(아직 로그 없음)"

def _clear_logs():
    """로그 버퍼 초기화"""
    _log_capture.clear()


# =========================================================
# 한글 폰트 설정
# =========================================================

def _setup_korean_font():
    import glob
    from matplotlib import font_manager as fm
    _sp.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], capture_output=True)
    _sp.run(['fc-cache', '-fv'], capture_output=True)
    for fpath in glob.glob('/usr/share/fonts/truetype/nanum/*.ttf'):
        fm.fontManager.addfont(fpath)
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in ['NanumGothic', 'Malgun Gothic', 'AppleGothic']:
        if name in installed:
            plt.rcParams['font.family'] = name
            print(f'  한글 폰트: {name}')
            return
    print('  한글 폰트를 찾지 못했습니다')

_setup_korean_font()
plt.rcParams['axes.unicode_minus'] = False


# =========================================================
# 공통 헬퍼
# =========================================================

def _kf(clip_id):
    d = os.path.join(PROJECT_ROOT, "data", "msrvtt", "keyframes")
    for ext in (".jpg", ".png", ".jpeg"):
        p = os.path.join(d, clip_id + ext)
        if os.path.exists(p):
            return p
    return None


def _extract_first_frame(video_path, save_path=None):
    """영상의 첫 프레임을 추출하여 반환"""
    try:
        cap = cv2.VideoCapture(video_path)
        ret, frame = cap.read()
        cap.release()
        if ret:
            if save_path:
                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                cv2.imwrite(save_path, frame)
            return frame
    except Exception:
        pass
    return None


def _get_video_duration_ms(video_path):
    """영상 길이(ms)를 반환"""
    try:
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        return (frames / fps) * 1000
    except Exception:
        return 5000

def _resolve_video_path(clip):
    """clip.video_path 경로 해석 — 없으면 VIDEO_DIR에서 clip_id.mp4 탐색"""
    if clip.video_path and os.path.exists(clip.video_path):
        return clip.video_path
    # fallback: VIDEO_DIR / {clip_id}.mp4
    fallback = os.path.join(VIDEO_DIR, f"{clip.clip_id}.mp4")
    if os.path.exists(fallback):
        clip.video_path = fallback  # 캐시 — 이후 재탐색 불필요
        return fallback
    return None

def _clip_preview(clip):
    """ClipResult에서 해당 구간만 잘라 미리보기 영상 생성"""
    if not clip:
        return None
    vpath = _resolve_video_path(clip)
    if not vpath:
        return None
    preview_dir = "output/preview"
    os.makedirs(preview_dir, exist_ok=True)
    out = os.path.join(preview_dir, f"{clip.clip_id}_preview.mp4")
    if os.path.exists(out):
        return out
    start_s = clip.start_ms / 1000
    end_s = clip.end_ms / 1000
    dur = end_s - start_s
    if dur <= 0:
        return vpath
    try:
        import subprocess
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start_s), "-i", vpath,
            "-t", str(dur), "-c:v", "libx264", "-preset", "ultrafast",
            "-an", "-loglevel", "error", out
        ], check=True, timeout=30)
        return out
    except Exception as e:
        _logging.getLogger().warning(f"클립 미리보기 생성 실패: {e}")
        return clip.video_path




N = gr.update


# ── 레이턴시 시각화 ──

PIPELINE_PHASES = [
    "phase0_preprocess", "phase1_bm25", "phase1_embed",
    "phase1_faiss", "phase2_fusion", "phase3_reranking", "phase3b_itm",
]

PHASE_LABELS = {
    "phase0_preprocess": "Phase 0: 쿼리 전처리",
    "phase1_bm25": "Phase 1: BM25 검색",
    "phase1_embed": "Phase 1: InternVideo2 임베딩",
    "phase1_faiss": "Phase 1: FAISS 검색",
    "phase2_fusion": "Phase 2: WRRF 퓨전",
    "phase3_reranking": "Phase 3: ColBERT 리랭킹",
    "phase3b_itm": "Phase 3b: ITM 재순위",
    "phase4_assembly": "Phase 4: 영상 합성",
}


def _make_latency_chart(latencies, title="레이턴시 분석"):
    if not latencies:
        return None
    items = sorted(latencies.items(), key=lambda x: x[1])
    labels = [PHASE_LABELS.get(k, k) for k, _ in items]
    values = [v for _, v in items]
    total = sum(values)
    fig, ax = plt.subplots(figsize=(7, max(1.8, len(labels) * 0.38)))
    colors = []
    for v in values:
        ratio = v / total if total else 0
        colors.append('#e74c3c' if ratio > 0.4 else '#f39c12' if ratio > 0.2 else '#2ecc71')
    bars = ax.barh(labels, values, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        pct = val / total * 100 if total else 0
        ax.text(bar.get_width() + total * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.0f}ms ({pct:.0f}%)', va='center', fontsize=8)
    ax.set_xlabel('ms')
    ax.set_title(f'{title}  (total: {total:.0f}ms)', fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return fig


# =========================================================
# 실시간 진행 표시 — 스레드 + 큐 + yield
# =========================================================

def _run_with_live_progress(fn, *args):
    """pipeline 함수를 스레드에서 실행하며 실시간 진행 yield"""
    progress_q = queue.Queue()

    def _cb(phase, status, ms):
        progress_q.put((phase, status, ms))

    pipeline._progress_callback = _cb
    result_holder = [None, None]

    def _run():
        try:
            result_holder[0] = fn(*args)
        except Exception as e:
            result_holder[1] = e

    t = threading.Thread(target=_run, daemon=True)
    t.start()

    completed = {}
    current_phase = None

    while t.is_alive():
        try:
            phase, status, ms = progress_q.get(timeout=0.25)
            if status == "start":
                current_phase = phase
            elif status == "done":
                completed[phase] = ms
                current_phase = None
        except queue.Empty:
            pass
        # 진행 텍스트 빌드
        lines = ["### 검색 진행 중...\n"]
        for p in PIPELINE_PHASES:
            label = PHASE_LABELS.get(p, p)
            if p in completed:
                lines.append(f"  ✅ {label} — {completed[p]:.0f}ms")
            elif p == current_phase:
                lines.append(f"  🔄 **{label}** 실행 중...")
            else:
                lines.append(f"  ⬜ {label}")
        yield "progress", "\n".join(lines)

    t.join()
    pipeline._progress_callback = None

    while not progress_q.empty():
        phase, status, ms = progress_q.get_nowait()
        if status == "done":
            completed[phase] = ms

    if result_holder[1]:
        yield "error", result_holder[1]
    else:
        yield "done", (result_holder[0], completed)


# =========================================================
# 클립 목록 관리 헬퍼 (Tab 2 용)
# =========================================================

def _clips_to_choices(state):
    """state["clips"]에서 CheckboxGroup choices 생성"""
    choices = []
    for c in state.get("clips", []):
        tag = {"archive": "📁", "transformed": "🔄", "generated": "🆕"}.get(c["source"], "")
        choices.append(f'{tag} {c["clip_id"]} ({c["score"]:.3f}) {c["caption"][:40]}')
    return choices


def _find_clip(state, choice_str):
    """CheckboxGroup 선택 문자열에서 clip_id를 찾아 clip dict 반환"""
    for c in state.get("clips", []):
        if c["clip_id"] in choice_str:
            return c
    return None


def _make_clip_result(clip_dict):
    """clip dict -> ClipResult 변환 (assembly용)"""
    return ClipResult(
        clip_id=clip_dict["clip_id"],
        score=clip_dict["score"],
        caption=clip_dict["caption"],
        video_path=clip_dict["video_path"],
        start_ms=clip_dict.get("start_ms", 0),
        end_ms=clip_dict.get("end_ms", _get_video_duration_ms(clip_dict["video_path"])),
        keyframe_path=clip_dict.get("keyframe_path", ""),
    )


# =========================================================
# =========================================================
# TAB 1: Scene Graph 워크플로 핸들러
# =========================================================
# =========================================================

SCENARIO_1 = {
    "title": "9시 뉴스: 서울 도심 야경 B-roll",
    "producer": "보도국 PD",
    "context": "강남역 상권 활성화 리포트 배경 영상",
    "scenes": [
        {"scene_id": 1,
         "description": "neon signs glowing on city buildings at night",
         "description_ko": "밤에 도시 건물들과 네온사인이 빛나는 거리",
         "duration_sec": 5,
         "attributes": {"time_of_day": "night", "mood": "cool", "location": "outdoor"}},
        {"scene_id": 2,
         "description": "people walking on a busy street at night",
         "description_ko": "사람들이 번화한 거리를 걸어다니는 모습",
         "duration_sec": 4,
         "attributes": {"time_of_day": "night", "mood": "bright", "location": "outdoor"}},
        {"scene_id": 3,
         "description": "cars driving on a road at night with headlights",
         "description_ko": "자동차들이 도로를 달리는 야간 장면",
         "duration_sec": 3,
         "attributes": {"time_of_day": "night", "mood": "dramatic", "location": "outdoor"}},
    ],
}

SCENARIO_2 = {
    "title": "예능: 셰프 요리 대결 하이라이트",
    "producer": "예능국 PD",
    "context": "요리 대결 프로그램 하이라이트 B-roll",
    "scenes": [
        {"scene_id": 1,
         "description": "a person cooking in a kitchen with a frying pan",
         "description_ko": "사람이 주방에서 요리하는 모습",
         "duration_sec": 6,
         "attributes": {"mood": "dramatic", "location": "indoor"}},
        {"scene_id": 2,
         "description": "close-up of food plated on a dish",
         "description_ko": "음식이 접시에 담겨있는 클로즈업",
         "duration_sec": 3,
         "attributes": {"mood": "warm", "location": "indoor"}},
        {"scene_id": 3,
         "description": "people eating food and reacting with excitement",
         "description_ko": "사람들이 음식을 먹으면서 반응하는 장면",
         "duration_sec": 4,
         "attributes": {"mood": "bright", "location": "indoor"}},
    ],
}

SCENARIO_3 = {
    "title": "다큐: 반려동물과 함께하는 도시인의 일상",
    "producer": "교양국 PD",
    "context": "반려동물 문화 다큐멘터리 인서트 영상",
    "scenes": [
        {"scene_id": 1,
         "description": "a dog running and playing in a park",
         "description_ko": "강아지가 공원에서 뛰어노는 장면",
         "duration_sec": 5,
         "attributes": {"time_of_day": "afternoon", "mood": "bright",
                        "season": "spring", "location": "outdoor"}},
        {"scene_id": 2,
         "description": "a cat resting comfortably indoors on a sofa",
         "description_ko": "고양이가 실내에서 편안하게 쉬고 있는 모습",
         "duration_sec": 4,
         "attributes": {"time_of_day": "afternoon", "mood": "warm",
                        "location": "indoor"}},
    ],
}

EXAMPLE_MAP = {
    "뉴스: 서울 야경 B-roll": json.dumps(SCENARIO_1, indent=2, ensure_ascii=False),
    "예능: 셰프 요리 대결": json.dumps(SCENARIO_2, indent=2, ensure_ascii=False),
    "다큐: 반려동물 일상": json.dumps(SCENARIO_3, indent=2, ensure_ascii=False),
}


def _run_prepare_with_progress(req, excluded_clip_ids=None):
    """mapper.prepare_scene()을 스레드에서 실행하며 실시간 진행 yield

    prepare_scene 내부에서 pipeline.search_only가 아닌 search_fn을 호출하므로
    pipeline._progress_callback 을 통해 Phase별 진행 상태를 받아온다.
    """
    progress_q = queue.Queue()

    def _cb(phase, status, ms):
        progress_q.put((phase, status, ms))

    pipeline._progress_callback = _cb
    result_holder = [None, None]

    def _run():
        try:
            result_holder[0] = mapper.prepare_scene(req, excluded_clip_ids=excluded_clip_ids)
        except Exception as e:
            result_holder[1] = e

    t = threading.Thread(target=_run, daemon=True)
    t.start()

    completed = {}
    current_phase = None

    while t.is_alive():
        try:
            phase, status, ms = progress_q.get(timeout=0.25)
            if status == "start":
                current_phase = phase
            elif status == "done":
                completed[phase] = ms
                current_phase = None
        except queue.Empty:
            pass
        yield "progress", _build_live_progress(completed, current_phase, req, excluded_clip_ids)

    t.join()
    pipeline._progress_callback = None

    while not progress_q.empty():
        phase, status, ms = progress_q.get_nowait()
        if status == "done":
            completed[phase] = ms

    if result_holder[1]:
        yield "error", result_holder[1]
    else:
        yield "done", (result_holder[0], completed)


def _build_live_progress(completed, current_phase, req=None, excluded_ids=None):
    """파이프라인 Phase별 실시간 진행 마크다운 빌드"""
    lines = ["### 검색 + 분기 판정 중...\n"]
    if req:
        lines.append(f"**검색 쿼리**: `{req.description[:80]}`")
        if excluded_ids:
            lines.append(f"**제외 클립**: {len(excluded_ids)}개")
        lines.append("")
    for p in PIPELINE_PHASES:
        label = PHASE_LABELS.get(p, p)
        if p in completed:
            lines.append(f"  ✅ {label} — {completed[p]:.0f}ms")
        elif p == current_phase:
            lines.append(f"  🔄 **{label}** 실행 중...")
        else:
            lines.append(f"  ⬜ {label}")
    return "\n".join(lines)


def _progress_md(idx, total, req, branch_str, extra=""):
    """장면 처리 진행 마크다운"""
    lines = [
        f"### 장면 {idx + 1}/{total}",
        f"- **설명**: {req.description}",
        f"- **분기**: `{branch_str}`",
    ]
    if extra:
        lines.append(extra)
    return "\n".join(lines)


def _out(state, progress="", scene_info="", before_img=None, after_img=None,
         prompt="", backends=None, scene_order="",
         prompt_visible=False, action_visible=False, next_visible=False,
         sort_visible=False, assemble_visible=False, result_video=None,
         latency_plot=None,
         clip_candidates=None, clip_select_video=None,
         clip_select_visible=False):
    """Tab 1 출력 19-요소 튜플 빌더

    순서: state, progress_md, scene_info_md, before_img, after_img,
          prompt_tb, backend_radio, exec_btn(visible), action_row(visible),
          next_btn(visible), scene_order_box, sort_row(visible),
          assemble_btn(visible), result_video, latency_plot,
          clip_candidates_radio, clip_select_video, clip_select_row(visible)
    """
    backend_choices = backends or []
    candidate_choices = clip_candidates or []
    return (
        state,
        progress,
        scene_info,
        before_img,
        after_img,
        prompt,
        gr.update(choices=backend_choices, value=backend_choices[0] if backend_choices else None),
        gr.update(),                            # prompt_col (always visible)
        gr.update(),                            # action_row (always visible)
        gr.update(),                            # next_btn (always visible)
        scene_order,
        gr.update(),                            # sort_row (always visible)
        gr.update(),                            # assemble_btn (always visible)
        result_video,
        latency_plot,
        gr.update(choices=candidate_choices, value=candidate_choices[0] if candidate_choices else None),  # clip radio
        clip_select_video,                     # clip preview video
        gr.update(),                              # clip_select_row (always visible)
        _get_logs(),  # 18: log_panel
    )


def _process(state):
    """장면 순회 제너레이터 — prepare_scene + UI 업데이트

    idx >= len(reqs) 이면 완료 요약 + scene_order_box 채워서 반환.
    아니면 실시간 진행 표시 후 리뷰 결과 반환.
    """
    reqs = state["requirements"]
    idx = state["current_idx"]
    total = len(reqs)

    # ── 모든 장면 완료 ──
    if idx >= total:
        mapped = state["mapped"]
        lines = []
        for m in mapped:
            branch_tag = m.branch.value.upper()
            clip_id = ""
            if m.selected_clip:
                clip_id = m.selected_clip.clip_id
            if m.transformed_video_path:
                clip_id = os.path.basename(m.transformed_video_path).replace(".mp4", "")
            lines.append(f"Scene {m.scene_id}: {m.requirement.description[:40]} — {clip_id} ({branch_tag})")
        scene_order_text = "\n".join(lines)

        # 모든 latency 병합
        all_lat = {}
        for k, v in state.get("all_latencies", {}).items():
            all_lat[k] = v

        summary = (
            f"### 모든 장면 처리 완료 ({total}개)\n\n"
            f"| Scene | 분기 | 클립 |\n|---|---|---|\n"
        )
        for m in mapped:
            cid = m.selected_clip.clip_id if m.selected_clip else "(없음)"
            if m.transformed_video_path:
                cid = os.path.basename(m.transformed_video_path).replace(".mp4", "")
            summary += f"| {m.scene_id} | {m.branch.value} | {cid} |\n"
        summary += "\n> 아래에서 장면 순서를 재배치한 뒤 **영상 합성**을 클릭하세요."

        yield _out(
            state, progress=summary, scene_order=scene_order_text,
            sort_visible=True, assemble_visible=True,
            latency_plot=_make_latency_chart(all_lat, "전체 레이턴시"),
        )
        return

    # ── 현재 장면 처리 ──
    req = reqs[idx]
    state["attempt"] = 1
    state["review"] = None
    state["execution"] = None

    # 실시간 진행 표시하면서 prepare_scene 실행
    review = None
    search_lat = {}

    # 이미 사용된 clip_id + 이전 장면 상위 후보 전체 제외 (검색 다양성 확보)
    used_clip_ids = set()
    for m in state.get("mapped", []):
        if m.selected_clip:
            used_clip_ids.add(m.selected_clip.clip_id)
        if m.top_candidates:
            for c in m.top_candidates:
                used_clip_ids.add(c.clip_id)

    for msg_type, msg_data in _run_prepare_with_progress(req, excluded_clip_ids=used_clip_ids):
        if msg_type == "progress":
            yield _out(state, progress=msg_data,
                       scene_info=f"### 장면 {idx + 1}/{total}\n{req.description}")
        elif msg_type == "error":
            yield _out(state,
                       progress=f"### 오류\n`{msg_data}`",
                       scene_info=f"장면 {idx + 1} 처리 실패")
            return
        elif msg_type == "done":
            review, search_lat = msg_data

    # 검색 레이턴시 누적
    for k, v in search_lat.items():
        key = f"scene{idx + 1}_{k}"
        state["all_latencies"][key] = v

    # 모든 분기에서 PD에게 상위 후보 클립을 보여줌 (클립 선택 가능)
    state["review"] = review
    branch_str = review.branch.value.upper()

    # 후보 클립 목록 (Radio 선택지)
    candidate_choices = []
    if review.top_candidates:
        for c in review.top_candidates:
            label = f"{c.clip_id} (score: {c.score:.4f})"
            candidate_choices.append(label)

    # 1순위 클립의 영상을 미리보기로 표시
    clip_preview_video = None
    if review.selected_clip:
        clip_preview_video = _resolve_video_path(review.selected_clip)

    before_img = _clip_preview(review.selected_clip)

    backends = review.available_backends or ["opencv"]

    extra_lines = []
    # 현재/목표 상태 정보
    if review.search_score:
        extra_lines.append(f"- **검색 점수**: {review.search_score:.4f}")
    if review.attribute_match_score:
        extra_lines.append(f"- **속성 일치도**: {review.attribute_match_score:.2f}")
    if review.current_state:
        cs = ", ".join(f"{k}={v}" for k, v in review.current_state.items())
        extra_lines.append(f"- **현재 상태**: {cs}")
    if review.target_state:
        ts = ", ".join(f"{k}={v}" for k, v in review.target_state.items())
        extra_lines.append(f"- **목표 상태**: {ts}")
    if review.selected_clip:
        extra_lines.append(f"- **선택 클립**: `{review.selected_clip.clip_id}`")
    extra_lines.append(f"\n> 프롬프트를 확인/수정하고 **실행**을 클릭하세요.")

    info = _progress_md(idx, total, req, branch_str, "\n".join(extra_lines))

    scene_chart = _make_latency_chart(search_lat, f"Scene {idx+1} 검색 레이턴시") if search_lat else None

    # USE_AS_IS면 프롬프트 불필요, 다음 장면 버튼만 표시
    is_use_as_is = (review.branch == BranchDecision.USE_AS_IS)

    yield _out(state, progress="", scene_info=info,
               before_img=before_img,
               prompt=review.auto_prompt if not is_use_as_is else "",
               backends=backends,
               prompt_visible=(not is_use_as_is),
               next_visible=is_use_as_is,
               latency_plot=scene_chart,
               clip_candidates=candidate_choices,
               clip_select_video=clip_preview_video,
               clip_select_visible=True,
               assemble_visible=len(state.get("mapped", [])) > 0,
               sort_visible=len(state.get("mapped", [])) > 0)


def on_load_example(example_name):
    """예시 시나리오 로드"""
    sg_json = EXAMPLE_MAP.get(example_name, "")
    return sg_json


def on_parse(sg_text, state):
    """Scene Graph JSON 파싱 시작 → _process 제너레이터 실행"""
    if not sg_text.strip():
        yield _out(state, progress="Scene Graph JSON을 입력하세요.")
        return

    try:
        sg = json.loads(sg_text)
    except json.JSONDecodeError as e:
        yield _out(state, progress=f"### JSON 파싱 오류\n`{e}`")
        return

    reqs = mapper.parse_scene_graph(sg)
    if not reqs:
        yield _out(state, progress="### 장면이 없습니다.\nScene Graph에 scenes 배열을 추가하세요.")
        return

    state["scenario"] = sg
    state["requirements"] = reqs
    state["current_idx"] = 0
    state["mapped"] = []
    state["review"] = None
    state["execution"] = None
    state["attempt"] = 1
    state["all_latencies"] = {}

    yield _out(state, progress=f"### Scene Graph 파싱 완료\n{len(reqs)}개 장면 발견. 처리를 시작합니다...")

    for out_tuple in _process(state):
        yield out_tuple


def on_execute(prompt, backend, state):
    """PD가 프롬프트를 확인/수정 후 실행"""
    # (로그 초기화하지 않음 — 이전 검색 로그 유지)
    review = state.get("review")
    if not review:
        yield _out(state, progress="리뷰 대상이 없습니다.")
        return

    attempt = state.get("attempt", 1)
    idx = state["current_idx"]
    total = len(state["requirements"])
    req = state["requirements"][idx]

    yield _out(state,
               progress=f"### 실행 중... (시도 {attempt}/3)\n백엔드: `{backend}`",
               scene_info=_progress_md(idx, total, req, review.branch.value.upper()))

    # -- working_video를 실행 대상으로 사용 --
    working = state.get("working_video")
    if working and os.path.exists(working) and review.selected_clip:
        review.selected_clip.video_path = working
        review.selected_clip.start_ms = 0
        review.selected_clip.end_ms = _get_video_duration_ms(working)

    if backend == "original":
        clip = review.selected_clip
        if clip and _resolve_video_path(clip):
            mapped_scene = MappedScene(
                scene_id=review.scene_id,
                requirement=review.requirement,
                selected_clip=clip,
                branch=BranchDecision.USE_AS_IS,
                search_score=review.search_score,
                attribute_match_score=review.attribute_match_score,
                candidates_count=len(review.top_candidates) if review.top_candidates else 0,
                notes="PD가 원본 영상 선택 (TRANSFORM → original)",
                top_candidates=review.top_candidates,
            )
            state["mapped"].append(mapped_scene)
            state["current_idx"] = state.get("current_idx", 0) + 1
            state["review"] = None
            state["execution"] = None
            for out_tuple in _process(state):
                yield out_tuple
            return
        else:
            yield _out(state, progress="### 원본 영상 파일을 찾을 수 없습니다.",
                       scene_info=_progress_md(idx, total, req, review.branch.value.upper()))
            return

    # ── 스레드로 실행 (Runway 1~5분 블로킹 대비, 로그 실시간 갱신) ──
    _exec_result = [None, None]  # [execution, exception]
    def _run_exec():
        try:
            _exec_result[0] = mapper.execute_scene(review, prompt, backend, attempt=attempt)
        except Exception as e:
            _exec_result[1] = e

    _t = threading.Thread(target=_run_exec, daemon=True)
    _t0 = time.perf_counter()
    _t.start()

    # 실행 중 로그 실시간 갱신
    while _t.is_alive():
        _elapsed = time.perf_counter() - _t0
        yield _out(state,
                   progress=f"### 실행 중... (시도 {attempt}/3, {_elapsed:.0f}초 경과)\n백엔드: `{backend}`",
                   scene_info=_progress_md(idx, total, req, review.branch.value.upper()),
                   before_img=state.get("working_video"))
        _t.join(timeout=2.0)  # 2초마다 UI 갱신

    elapsed_ms = (time.perf_counter() - _t0) * 1000

    if _exec_result[1]:
        yield _out(state, progress=f"### 실행 오류\n`{_exec_result[1]}`",
                   scene_info=_progress_md(idx, total, req, review.branch.value.upper()),
                   prompt=prompt, backends=review.available_backends,
                   prompt_visible=True)
        return

    execution = _exec_result[0]

    state["execution"] = execution
    state["all_latencies"][f"scene{idx + 1}_execute"] = elapsed_ms

    after_img = execution.output_path if execution.success else execution.after_keyframe_path
    before_img = state.get("working_video") or _clip_preview(review.selected_clip)

    if execution.success:
        info = (
            _progress_md(idx, total, req, review.branch.value.upper(),
                         f"- **실행 성공** ({elapsed_ms:.0f}ms)\n"
                         f"- **백엔드**: `{backend}`\n"
                         f"- **시도**: {attempt}/3\n\n"
                         f"> **승인**, **재시도**, **건너뛰기**, **업로드** 중 선택하세요.")
        )
        yield _out(state, scene_info=info,
                   before_img=before_img, after_img=after_img,
                   action_visible=True)
    else:
        err = execution.error or "원인 불명 (execution.error=None)"
        info = (
            _progress_md(idx, total, req, review.branch.value.upper(),
                         f"- **실행 실패**: `{err}`\n"
                         f"- **백엔드**: `{backend}`\n"
                         f"- **시도**: {attempt}/3\n"
                         f"- **output_path**: `{getattr(execution, 'output_path', None)}`\n\n"
                         f"> **재시도**, **건너뛰기**, **업로드** 중 선택하세요.")
        )
        yield _out(state, scene_info=info,
                   before_img=before_img,
                   prompt=prompt, backends=review.available_backends,
                   prompt_visible=True, action_visible=True)


def on_accept(state):
    """PD가 실행 결과를 승인 → MappedScene 확정 → 다음 장면"""
    _clear_logs()
    review = state.get("review")
    execution = state.get("execution")
    if not review or not execution:
        yield _out(state, progress="승인할 결과가 없습니다.")
        return

    mapped_scene = mapper.accept_scene(review, execution)
    state["mapped"].append(mapped_scene)
    state["current_idx"] += 1
    state["review"] = None
    state["execution"] = None

    for out_tuple in _process(state):
        yield out_tuple


def on_retry(state):
    """재시도 → attempt 증가 후 프롬프트 편집 상태로 복귀"""
    state["attempt"] = state.get("attempt", 1) + 1
    if state["attempt"] > 3:
        # 3회 초과 → 건너뛰기 제안
        yield _out(state, progress="### 3회 재시도 초과\n**건너뛰기** 또는 **업로드**를 선택하세요.",
                   action_visible=True)
        return

    review = state.get("review")
    if not review:
        yield _out(state, progress="리뷰 대상이 없습니다.")
        return

    idx = state["current_idx"]
    total = len(state["requirements"])
    req = state["requirements"][idx]

    info = _progress_md(idx, total, req, review.branch.value.upper(),
                        f"- 🔄 재시도 준비 (시도 {state['attempt']}/3)\n"
                        f"> 프롬프트를 수정하고 **실행**을 클릭하세요.")

    # working_video(crop된 영상)가 있으면 before로 복원, 없으면 keyframe 사용
    before = state.get("working_video") or review.before_keyframe_path

    yield _out(state, scene_info=info,
               before_img=before,
               prompt=review.auto_prompt,
               backends=review.available_backends,
               prompt_visible=True)


def on_skip(state):
    """장면 건너뛰기"""
    review = state.get("review")
    if not review:
        yield _out(state, progress="건너뛸 대상이 없습니다.")
        return

    mapped_scene = mapper.skip_scene(review)
    state["mapped"].append(mapped_scene)
    state["current_idx"] += 1
    state["review"] = None
    state["execution"] = None

    for out_tuple in _process(state):
        yield out_tuple


def on_upload(file_obj, state):
    """PD가 직접 파일을 업로드하여 장면 확정"""
    review = state.get("review")
    if not review:
        yield _out(state, progress="업로드할 대상이 없습니다.")
        return

    if file_obj is None:
        yield _out(state, progress="파일을 선택하세요.", action_visible=True)
        return

    # Gradio File 객체에서 경로 추출
    uploaded_path = file_obj.name if hasattr(file_obj, 'name') else str(file_obj)

    # 출력 디렉토리에 복사
    os.makedirs("output/uploaded", exist_ok=True)
    dest = f"output/uploaded/scene{review.scene_id}_upload.mp4"
    import shutil
    shutil.copy2(uploaded_path, dest)

    mapped_scene = mapper.accept_upload(review, dest)
    state["mapped"].append(mapped_scene)
    state["current_idx"] += 1
    state["review"] = None
    state["execution"] = None

    for out_tuple in _process(state):
        yield out_tuple


def on_select_clip(clip_choice, state):
    """PD가 후보 클립 선택 -> 미리보기 + Before + 크롭 슬라이더 업데이트"""
    review = state.get("review")
    if not review or not review.top_candidates or not clip_choice:
        return state, None, None, gr.update(), gr.update(), gr.update(visible=False), ""

    selected_id = clip_choice.split(" (score:")[0].strip()
    for c in review.top_candidates:
        if c.clip_id == selected_id:
            review.selected_clip = c
            review.search_score = c.score
            state["review"] = review
            state["crop_video_path"] = None
            clip_video = c.video_path if (c.video_path and os.path.exists(c.video_path)) else None
            preview = _clip_preview(c)
            state["working_video"] = preview
            dur = (c.end_ms - c.start_ms) / 1000
            if dur <= 0:
                dur = 10.0
            if not clip_video:
                _logging.getLogger().warning(f"클립 {c.clip_id} 영상 파일 없음: {c.video_path}")
            return (
                state, clip_video, preview,
                gr.update(minimum=0, maximum=dur, value=0, step=0.1),
                gr.update(minimum=0, maximum=dur, value=min(5, dur), step=0.1),
                gr.update(visible=True),
                f"클립 길이: {dur:.1f}초",
            )

    return state, None, None, gr.update(), gr.update(), gr.update(visible=False), ""


def on_crop(start, end, state):
    """PD가 선택한 구간을 크롭"""
    review = state.get("review")
    if not review or not review.selected_clip:
        return state, None, "클립을 먼저 선택하세요"
    c = review.selected_clip
    if not c.video_path or not os.path.exists(c.video_path):
        return state, None, "영상 파일 없음"
    abs_start = c.start_ms / 1000 + start
    dur = end - start
    if dur <= 0:
        return state, None, "구간 유효하지 않음 (끝 > 시작)"
    crop_path = f"output/preview/{c.clip_id}_crop.mp4"
    os.makedirs("output/preview", exist_ok=True)
    try:
        import subprocess
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(abs_start), "-i", c.video_path,
            "-t", str(dur), "-c:v", "libx264", "-preset", "ultrafast",
            "-an", "-loglevel", "error", crop_path
        ], check=True, timeout=30)
    except Exception as e:
        return state, None, f"크롭 실패: {e}"
    state["crop_video_path"] = crop_path
    state["working_video"] = crop_path
    _logging.getLogger().info(f"구간 크롭: {start:.1f}s~{end:.1f}s -> {crop_path}")
    return state, crop_path, f"크롭 완료: {dur:.1f}초 구간"


def on_crop_full(state):
    """전체 클립 사용 (크롭 안 함)"""
    review = state.get("review")
    if not review or not review.selected_clip:
        return state, None, "클립을 먼저 선택하세요"
    state["crop_video_path"] = None
    state["working_video"] = _clip_preview(review.selected_clip)
    preview = _clip_preview(review.selected_clip)
    return state, preview, "전체 클립 사용"


def on_next(state):
    """USE_AS_IS 후 다음 장면으로 이동 (PD가 선택한 클립으로 확정)"""
    review = state.get("review")
    if review:
        mapped_scene = MappedScene(
            scene_id=review.scene_id,
            requirement=review.requirement,
            selected_clip=review.selected_clip,
            branch=BranchDecision.USE_AS_IS,
            search_score=review.search_score,
            attribute_match_score=review.attribute_match_score,
            candidates_count=len(review.top_candidates) if review.top_candidates else 0,
            notes="PD가 클립 선택 후 확정 (USE_AS_IS)",
            top_candidates=review.top_candidates,
        )
        state["mapped"].append(mapped_scene)
        state["current_idx"] = state.get("current_idx", 0) + 1
        state["review"] = None
        state["execution"] = None

    for out_tuple in _process(state):
        yield out_tuple


def on_assemble_sg(scene_order_text, sort_mode, state):
    """Scene Graph 워크플로 최종 합성 — scene_order_box 순서대로 합성"""
    _clear_logs()
    mapped = state.get("mapped", [])
    if not mapped:
        return state, "처리된 장면이 없습니다.", None, None, _get_logs()

    try:
        return _assemble_sg_inner(scene_order_text, sort_mode, state, mapped)
    except Exception as e:
        import traceback
        err_detail = traceback.format_exc()
        err_msg = (
            f"### ❌ 합성 오류\n\n"
            f"**에러**: `{str(e)}`\n\n"
            f"```\n{err_detail}\n```"
        )
        return state, err_msg, None, None, _get_logs()


def _assemble_sg_inner(scene_order_text, sort_mode, state, mapped):
    """on_assemble_sg 내부 로직 (에러 핸들링 분리)"""
    _clear_logs()

    # scene_order_text에서 순서 파악
    lines = [l.strip() for l in scene_order_text.strip().split("\n") if l.strip()]

    if lines:
        # 라인에서 Scene ID 추출하여 재정렬
        ordered = []
        for line in lines:
            for m in mapped:
                scene_tag = f"Scene {m.scene_id}"
                if scene_tag in line and m not in ordered:
                    ordered.append(m)
                    break
        if not ordered:
            logger.warning("순서 파싱 실패 — 원래 순서 사용")
            ordered = mapped
    else:
        ordered = mapped

    # MappedScene -> ClipResult 리스트로 변환
    clip_results = []
    for m in ordered:
        if m.transformed_video_path and os.path.exists(m.transformed_video_path):
            cr = ClipResult(
                clip_id=f"scene{m.scene_id}_transformed",
                score=m.search_score,
                caption=m.requirement.description[:80],
                video_path=m.transformed_video_path,
                start_ms=0,
                end_ms=_get_video_duration_ms(m.transformed_video_path),
            )
            clip_results.append(cr)
        elif m.selected_clip and m.selected_clip.video_path:
            clip_results.append(m.selected_clip)
        # 건너뛴 장면(클립 없음)은 제외

    if not clip_results:
        return state, "유효한 클립이 없습니다.", None, None, _get_logs()

    curation = CurationState(
        search_results=clip_results,
        selected_clip_ids=[c.clip_id for c in clip_results],
        ordering=[c.clip_id for c in clip_results],
        excluded_ids=set(),
    )

    t0 = time.perf_counter()
    result = pipeline.assemble_curated(curation, query=state.get("scenario", {}).get("title", ""), sort_mode=sort_mode)
    assemble_ms = (time.perf_counter() - t0) * 1000

    state["all_latencies"]["phase4_assembly"] = assemble_ms

    out_md = (
        f"### 합성 완료\n\n"
        f"- **클립**: {len(clip_results)}개\n"
        f"- **합성 시간**: {assemble_ms:.0f}ms\n"
        f"- **TC-Score**: {f'{result.tc_score:.4f}' if result.tc_score is not None else 'N/A'}\n"
    )

    if result.c2pa_manifest and result.c2pa_manifest.signature:
        out_md += f"\n**C2PA 서명**: `{result.c2pa_manifest.signature[:40]}...`"

    vp = result.video_path if result.video_path and os.path.exists(result.video_path) else None

    all_lat = state.get("all_latencies", {})
    all_lat["phase4_assembly"] = assemble_ms
    chart = _make_latency_chart(all_lat, "Scene Graph 워크플로 레이턴시")

    return state, out_md, vp, chart, _get_logs()


# =========================================================
# =========================================================
# TAB 2: PD 큐레이션 핸들러
# =========================================================
# =========================================================

def on_search(query, top_k, state):
    """검색 실행 (누적 모드) -> 기존 클립 유지 + 새 결과 추가"""
    if not query.strip():
        yield state, "쿼리를 입력하세요.", "", N(), None, N()
        return

    curation = None
    search_lat = {}

    for msg_type, msg_data in _run_with_live_progress(pipeline.search_only, query, int(top_k)):
        if msg_type == "progress":
            yield state, msg_data, "", N(), None, N()
        elif msg_type == "error":
            yield state, f"### 오류\n`{msg_data}`", "", N(), None, N()
            return
        elif msg_type == "done":
            curation, search_lat = msg_data

    # 새 검색 → 기존 클립 리셋
    clips = []
    for c in curation.search_results:
        clips.append({
            "clip_id": c.clip_id, "source": "archive",
            "score": c.score, "caption": c.caption,
            "video_path": c.video_path,
            "start_ms": c.start_ms, "end_ms": c.end_ms,
            "keyframe_path": c.keyframe_path or _kf(c.clip_id),
        })

    state = {
        "query": query, "curation": curation,
        "clips": clips, "search_latencies": search_lat,
        "transform_target": None, "gen_count": 0,
    }

    total = sum(search_lat.values()) if search_lat else 0
    out = f"### 검색 결과: {len(clips)}개 클립 ({total:.0f}ms)\n\n"
    out += f"쿼리: `{query}`\n\n"
    out += "| # | clip_id | score | caption |\n|---|---------|-------|---------|\n"
    for i, c in enumerate(clips, 1):
        out += f"| {i} | {c['clip_id']} | {c['score']:.4f} | {c['caption'][:50]} |\n"
    out += "\n> 클립을 선택하세요. 선택 후 **변환**, **제거**, 또는 **영상 합성**이 가능합니다."

    choices = _clips_to_choices(state)
    chart = _make_latency_chart(search_lat, "검색 파이프라인 레이턴시")
    yield state, out, "", gr.update(choices=choices), chart, N()


def on_reset_clips(state):
    """클립 목록 초기화"""
    state["clips"] = []
    state["transform_target"] = None
    state["gen_count"] = 0
    return state, gr.update(choices=[], value=[]), "클립 목록 초기화됨.", "", N()


def on_clip_detail(selected_clip, state):
    """클립 선택 시 영상 미리보기"""
    if not selected_clip:
        return None
    clip = _find_clip(state, selected_clip)
    if not clip:
        return None
    return clip.get("video_path")

def on_open_transform(selected_clip, state):
    """변환 패널 열기"""
    if not selected_clip:
        return state, N(), None, None, "", N(), gr.update(visible=False), ""
    clip = _find_clip(state, selected_clip)
    if not clip:
        return state, N(), None, None, "", N(), gr.update(visible=False), "클립을 찾을 수 없습니다."

    state["transform_target"] = clip["clip_id"]
    state["transform_attempt"] = 1

    # Before 키프레임
    before = clip.get("keyframe_path") or _kf(clip["clip_id"])
    if not before and clip["video_path"]:
        frame = _extract_first_frame(clip["video_path"])
        if frame is not None:
            before_path = f"output/preview/{clip['clip_id']}_before.jpg"
            os.makedirs("output/preview", exist_ok=True)
            cv2.imwrite(before_path, frame)
            before = before_path

    # 기본 프롬프트
    default_prompt = f"Transform this video clip: {clip['caption'][:80]}"

    # runway API 키가 있으면 runway 옵션 추가 (video-to-video)
    has_runway = bool(getattr(inverse_engine, "runway_api_key", None))
    backends = ["runway", "opencv"] if has_runway else ["opencv"]
    default_backend = backends[0]

    return (state, gr.update(visible=True), before, None, default_prompt,
            gr.update(choices=backends, value=default_backend),
            gr.update(visible=True), "")


def on_exec_transform(prompt, backend, state):
    """변환 실행"""
    clip_id = state.get("transform_target")
    clip = None
    for c in state.get("clips", []):
        if c["clip_id"] == clip_id:
            clip = c
            break
    if not clip:
        return state, None, "클립을 찾을 수 없습니다.", gr.update(visible=False), gr.update(visible=False)

    attempt = state.get("transform_attempt", 1)
    output_path = f"output/transformed/{clip_id}_T{attempt}.mp4"
    os.makedirs("output/transformed", exist_ok=True)

    try:
        t0 = time.perf_counter()
        result = inverse_engine.apply_transform(
            video_path=clip["video_path"],
            prompt=prompt,
            output_path=output_path,
            backend=backend,
        )
        elapsed = (time.perf_counter() - t0) * 1000
    except Exception as e:
        return state, None, f"변환 실패: {e}", gr.update(visible=False), gr.update(visible=False)

    # Runway 등 API 실패 시 에러 메시지 표시
    if not result.get("success", True):
        error_msg = result.get("error", "알 수 없는 오류")
        return (state, None, f"### 변환 실패\n\n**백엔드**: `{backend}`\n**에러**: `{error_msg}`\n\n> 다른 백엔드를 선택하거나 프롬프트를 수정하세요.", gr.update(visible=False), gr.update(visible=False))

    # After → 영상으로 표시 (gr.Video 변경)
    after_video = None
    out_path = result.get("output_path", output_path)
    if out_path and os.path.exists(out_path):
        after_video = out_path
        state["_last_transform_output"] = out_path

    status = f"### 변환 완료 ({elapsed:.0f}ms)\n백엔드: `{backend}`\n\n> **승인** 또는 **재시도**를 선택하세요."
    return state, after_video, status, gr.update(visible=True), gr.update(visible=True)


def on_approve_transform(state):
    """변환 결과 승인 -> 클립 목록에 추가"""
    clip_id = state.get("transform_target")
    out_path = state.get("_last_transform_output")
    if not clip_id or not out_path:
        return state, N(), gr.update(visible=False), "승인할 결과가 없습니다."

    original = None
    for c in state.get("clips", []):
        if c["clip_id"] == clip_id:
            original = c
            break

    new_id = f"{clip_id}_T{state.get('transform_attempt', 1)}"
    new_clip = {
        "clip_id": new_id, "source": "transformed",
        "score": 1.0,
        "caption": f"[변환] {original['caption'][:60]}" if original else "[변환됨]",
        "video_path": out_path,
        "start_ms": original.get("start_ms", 0) if original else 0,
        "end_ms": original.get("end_ms", 5000) if original else 5000,
        "keyframe_path": state.get("_last_transform_output", "").replace(".mp4", "_after.jpg"),
    }
    state["clips"].append(new_clip)
    state["transform_target"] = None

    choices = _clips_to_choices(state)
    return state, gr.update(choices=choices, value=[ch for ch in choices if "📁" in ch or "🔄" in ch or "🆕" in ch][:10]), gr.update(visible=False), f"✅ `{new_id}` 추가됨"


def on_retry_transform(state):
    """변환 재시도"""
    state["transform_attempt"] = state.get("transform_attempt", 1) + 1
    if state["transform_attempt"] > 3:
        return state, "", gr.update(visible=False), gr.update(visible=False)
    return state, f"🔄 재시도 준비 (시도 {state['transform_attempt']}/3). 프롬프트를 수정하고 **실행**을 클릭하세요.", N(), N()


def on_cancel_transform(state):
    """변환 취소"""
    state["transform_target"] = None
    return state, gr.update(visible=False), ""


def on_generate(prompt, duration, backend, state):
    """새 클립 생성"""
    if not prompt.strip():
        yield state, "프롬프트를 입력하세요.", None, N()
        return

    state["gen_count"] = state.get("gen_count", 0) + 1
    gen_id = f"GEN_{state['gen_count']:03d}"
    output_path = f"output/generated/{gen_id}.mp4"
    os.makedirs("output/generated", exist_ok=True)

    yield state, f"### 생성 중...\n프롬프트: `{prompt[:80]}`\n백엔드: `{backend}`", None, N()

    try:
        t0 = time.perf_counter()
        result = inverse_engine.generate_video(
            prompt=prompt,
            duration_sec=float(duration),
            output_path=output_path,
            backend=backend,
        )
        elapsed = (time.perf_counter() - t0) * 1000
    except Exception as e:
        yield state, f"### 생성 실패\n`{e}`", None, N()
        return

    out_path = result.get("output_path", output_path)

    kf_path = f"output/keyframes/{gen_id}.jpg"
    _extract_first_frame(out_path, kf_path)

    new_clip = {
        "clip_id": gen_id, "source": "generated",
        "score": 1.0, "caption": f"[생성] {prompt[:60]}",
        "video_path": out_path,
        "start_ms": 0,
        "end_ms": float(duration) * 1000,
        "keyframe_path": kf_path if os.path.exists(kf_path) else "",
    }
    state["clips"].append(new_clip)

    choices = _clips_to_choices(state)
    vp = out_path if os.path.exists(out_path) else None
    status = f"### 생성 완료 ({elapsed:.0f}ms)\n`{gen_id}` 추가됨. 클립 목록에서 선택하세요."
    yield state, status, vp, gr.update(choices=choices, value=[ch for ch in choices][:10])


def on_remove_clip(selected_detail, state):
    """선택된 클립을 목록에서 제거"""
    if not selected_detail:
        return state, N(), ""
    clip = _find_clip(state, selected_detail)
    if clip:
        state["clips"] = [c for c in state["clips"] if c["clip_id"] != clip["clip_id"]]
    choices = _clips_to_choices(state)
    return state, gr.update(choices=choices), f"✅ 제거됨"


def on_assemble(order_text, selected_clips, sort_mode, state):
    """선택된 클립들로 영상 합성 (manual: order_box 순서 사용)"""
    if not selected_clips and not order_text.strip():
        return state, "클립을 선택하세요.", None, None

    # manual 모드: order_box의 줄 순서 사용
    if sort_mode == "manual" and order_text.strip():
        lines = [l.strip() for l in order_text.strip().split("\n") if l.strip()]
        clip_results = []
        for line in lines:
            clip = _find_clip(state, line)
            if clip:
                clip_results.append(_make_clip_result(clip))
    else:
        clip_results = []
        for choice in (selected_clips or []):
            clip = _find_clip(state, choice)
            if clip:
                clip_results.append(_make_clip_result(clip))

    if not clip_results:
        return state, "유효한 클립이 없습니다.", None, None

    curation = CurationState(
        search_results=clip_results,
        selected_clip_ids=[c.clip_id for c in clip_results],
        ordering=[c.clip_id for c in clip_results],
        excluded_ids=set(),
    )

    query = state.get("query", "")
    t0 = time.perf_counter()
    result = pipeline.assemble_curated(curation, query=query, sort_mode=sort_mode)
    assemble_ms = (time.perf_counter() - t0) * 1000

    out = (
        "### 합성 완료\n\n"
        f"- **클립**: {len(clip_results)}개 (📁 아카이브 {sum(1 for c in selected_clips if '📁' in c)} "
        f"/ 🔄 변환 {sum(1 for c in selected_clips if '🔄' in c)} "
        f"/ 🆕 생성 {sum(1 for c in selected_clips if '🆕' in c)})\n"
        f"- **레이턴시**: {assemble_ms:.0f}ms\n"
        f"- **TC-Score**: {f'{result.tc_score:.4f}' if result.tc_score is not None else 'N/A'}\n"
    )

    if result.c2pa_manifest and result.c2pa_manifest.signature:
        out += f"\n\n**C2PA 서명**: `{result.c2pa_manifest.signature[:40]}...`"

    vp = result.video_path if result.video_path and os.path.exists(result.video_path) else None
    chart = _make_latency_chart(result.phase_latencies, "영상 합성 레이턴시") if result.phase_latencies else None
    return state, out, vp, chart


# =========================================================
# =========================================================
# Gradio UI 구성
# =========================================================
# =========================================================

with gr.Blocks(
    title="VideoRAG PD 워크스테이션 v4",
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown(
        "# 🎬 VideoRAG — PD 워크스테이션 v4\n"
        "Scene Graph 자동 워크플로 + PD 큐레이션 수동 워크플로"
    )

    # ==============================================================
    # TAB 1: Scene Graph 워크플로
    # ==============================================================
    with gr.Tab("Scene Graph 워크플로"):

        wf_state = gr.State({
            "scenario": {}, "requirements": [], "current_idx": 0,
            "mapped": [], "review": None, "execution": None,
            "attempt": 1, "all_latencies": {},
        })

        gr.Markdown("## 📋 Scene Graph 입력")
        with gr.Row():
            with gr.Column(scale=1):
                sg_example_dd = gr.Dropdown(
                    choices=["뉴스: 서울 야경 B-roll", "예능: 셰프 요리 대결", "다큐: 반려동물 일상"],
                    label="예시 시나리오",
                    value=None,
                    interactive=True,
                )
                sg_code = gr.Code(
                    language="json",
                    label="Scene Graph JSON",
                    lines=15,
                )
                sg_parse_btn = gr.Button("🎬 파싱 시작", variant="primary", size="lg")
            with gr.Column(scale=2):
                sg_progress_md = gr.Markdown("")
                sg_scene_info_md = gr.Markdown("")

        gr.Markdown("---")
        gr.Markdown("## 🔍 장면 처리")

        # ── 후보 클립 선택 (PD가 상위 5개 중 선택) ──
        with gr.Row() as sg_clip_select_row:
            with gr.Column(scale=1):
                sg_clip_radio = gr.Radio(
                    label="후보 클립 선택 (상위 5개)",
                    choices=[],
                    interactive=True,
                )
            with gr.Column(scale=2):
                sg_clip_preview = gr.Video(label="클립 미리보기", height=240)


        # -- 구간 크롭 (PD가 클립에서 원하는 구간 선택) --
        with gr.Row() as sg_crop_row:
            with gr.Column(scale=2):
                sg_crop_start = gr.Slider(
                    0, 30, value=0, step=0.1,
                    label="시작 (초)", interactive=True,
                )
                sg_crop_end = gr.Slider(
                    0, 30, value=5, step=0.1,
                    label="끝 (초)", interactive=True,
                )
            with gr.Column(scale=1):
                sg_crop_info = gr.Markdown("구간을 선택하세요")
                sg_crop_btn = gr.Button("구간 크롭", variant="secondary")
                sg_crop_full_btn = gr.Button("전체 사용", variant="secondary")

        with gr.Row():
            sg_before_img = gr.Video(label="Before", height=200)
            sg_after_img = gr.Video(label="After", height=200)

        # 프롬프트 편집 + 실행 (TRANSFORM/GENERATE 시)
        with gr.Column() as sg_prompt_col:
            sg_prompt_tb = gr.Textbox(label="프롬프트 (수정 가능)", lines=3)
            sg_backend_radio = gr.Radio(label="백엔드", choices=["opencv"])
            sg_exec_btn = gr.Button("▶️ 실행", variant="primary")

        # 액션 버튼 (승인/재시도/건너뛰기/업로드)
        with gr.Row() as sg_action_row:
            sg_accept_btn = gr.Button("✅ 승인", variant="primary")
            sg_retry_btn = gr.Button("🔄 재시도")
            sg_skip_btn = gr.Button("⏭ 건너뛰기")
            sg_upload_file = gr.File(label="📤 업로드", file_types=[".mp4", ".mov", ".avi"])
            sg_upload_btn = gr.Button("📤 업로드 확정")

        # USE_AS_IS용 다음 장면 버튼
        sg_next_btn = gr.Button("▶ 다음 장면", variant="secondary")

        gr.Markdown("---")
        gr.Markdown("## 🔀 장면 재배치 + 합성")

        sg_scene_order_box = gr.Textbox(
            label="장면 순서 (줄 순서 = 영상 순서, 직접 수정하여 재배치)",
            lines=8,
            placeholder="모든 장면 처리 완료 후 여기에 순서가 표시됩니다.",
            interactive=True,
        )

        with gr.Row() as sg_sort_row:
            sg_sort_radio = gr.Radio(
                choices=["manual", "auto", "hybrid"], value="manual",
                label="정렬 방식",
                info="manual: 위 순서 그대로 / auto: 시각 유사도 자동 / hybrid: 혼합",
            )

        sg_assemble_btn = gr.Button("🎬 영상 합성", variant="primary", size="lg")

        sg_assemble_md = gr.Markdown("")
        with gr.Row():
            sg_result_video = gr.Video(label="합성 영상")
            sg_latency_plot = gr.Plot(label="레이턴시 분석")

        gr.Markdown("---")
        gr.Markdown("## 📋 실시간 로그")
        sg_log_panel = gr.Textbox(
            label="파이프라인 로그 (에러/경고/진행)",
            lines=15,
            interactive=False,
            show_copy_button=True,
        )


        # ── Tab 1 출력 리스트 (18개: _out 함수와 일치) ──
        tab1_outs = [
            wf_state,           # 0: state
            sg_progress_md,     # 1: progress
            sg_scene_info_md,   # 2: scene_info
            sg_before_img,      # 3: before_img
            sg_after_img,       # 4: after_img
            sg_prompt_tb,       # 5: prompt
            sg_backend_radio,   # 6: backend_radio
            sg_prompt_col,      # 7: exec area visible (prompt_col)
            sg_action_row,      # 8: action_row visible
            sg_next_btn,        # 9: next_btn visible
            sg_scene_order_box, # 10: scene_order_box
            sg_sort_row,        # 11: sort_row visible
            sg_assemble_btn,    # 12: assemble_btn visible
            sg_result_video,    # 13: result_video
            sg_latency_plot,    # 14: latency_plot
            sg_clip_radio,      # 15: clip candidates radio
            sg_clip_preview,    # 16: clip preview video
            sg_clip_select_row, # 17: clip select row visible
            sg_log_panel,         # 18: log_panel
        ]

        # ── Tab 1 이벤트 연결 ──

        sg_example_dd.change(
            fn=on_load_example,
            inputs=[sg_example_dd],
            outputs=[sg_code],
        )

        sg_parse_btn.click(
            fn=on_parse,
            inputs=[sg_code, wf_state],
            outputs=tab1_outs,
        )

        sg_exec_btn.click(
            fn=on_execute,
            inputs=[sg_prompt_tb, sg_backend_radio, wf_state],
            outputs=tab1_outs,
        )

        sg_accept_btn.click(
            fn=on_accept,
            inputs=[wf_state],
            outputs=tab1_outs,
        )

        sg_retry_btn.click(
            fn=on_retry,
            inputs=[wf_state],
            outputs=tab1_outs,
        )

        sg_skip_btn.click(
            fn=on_skip,
            inputs=[wf_state],
            outputs=tab1_outs,
        )

        sg_upload_btn.click(
            fn=on_upload,
            inputs=[sg_upload_file, wf_state],
            outputs=tab1_outs,
        )

        sg_clip_radio.change(
            fn=on_select_clip,
            inputs=[sg_clip_radio, wf_state],
            outputs=[wf_state, sg_clip_preview, sg_before_img, sg_crop_start, sg_crop_end, sg_crop_row, sg_crop_info],
        )

        sg_crop_btn.click(
            fn=on_crop,
            inputs=[sg_crop_start, sg_crop_end, wf_state],
            outputs=[wf_state, sg_before_img, sg_crop_info],
        )

        sg_crop_full_btn.click(
            fn=on_crop_full,
            inputs=[wf_state],
            outputs=[wf_state, sg_before_img, sg_crop_info],
        )

        sg_next_btn.click(
            fn=on_next,
            inputs=[wf_state],
            outputs=tab1_outs,
        )

        sg_assemble_btn.click(
            fn=on_assemble_sg,
            inputs=[sg_scene_order_box, sg_sort_radio, wf_state],
            outputs=[wf_state, sg_assemble_md, sg_result_video, sg_latency_plot, sg_log_panel],
        )

    # ==============================================================
    # TAB 2: PD 큐레이션
    # ==============================================================
    with gr.Tab("PD 큐레이션"):

        ws_state = gr.State({})

        # ── Section A: 검색 ──
        gr.Markdown("## 🔍 검색")
        with gr.Row():
            with gr.Column(scale=1):
                cur_query = gr.Textbox(label="검색 쿼리", placeholder="예: a dog running in a park", lines=2)
                cur_topk = gr.Slider(1, 20, value=10, step=1, label="Top K")
                search_btn = gr.Button("🔍 검색", variant="primary")
            with gr.Column(scale=2):
                search_progress_md = gr.Markdown("")
                search_latency_plot = gr.Plot(label="검색 레이턴시")

        # ── Section B: 클립 선택 + 액션 ──
        gr.Markdown("---")
        gr.Markdown("## 📋 클립 선택")
        with gr.Row():
            with gr.Column(scale=1):
                clips_cbg = gr.CheckboxGroup(choices=[], label="클립 선택 (체크 순서 = 배치 순서)")
                clip_detail_dd = gr.Dropdown(choices=[], label="미리보기 클립", interactive=True, visible=False)
            with gr.Column(scale=2):
                clip_preview_video = gr.Video(label="클립 미리보기", height=200)

        # ── Section C: 변환 패널 (비활성) ──
        with gr.Column(visible=False) as transform_col:
            gr.Markdown("### 🔄 클립 변환")
            with gr.Row():
                before_img = gr.Video(label="Before", height=200)
                after_img = gr.Video(label="After", height=200)
            transform_prompt = gr.Textbox(label="변환 프롬프트 (수정 가능)", lines=3)
            transform_backend = gr.Radio(label="백엔드", choices=[])
            with gr.Row():
                exec_transform_btn = gr.Button("▶️ 실행", variant="primary")
                cancel_transform_btn = gr.Button("취소")
            transform_status_md = gr.Markdown("")
            with gr.Row(visible=False) as transform_action_row:
                approve_transform_btn = gr.Button("✅ 승인", variant="primary")
                retry_transform_btn = gr.Button("🔄 재시도")

        # ── Section D: 새 클립 생성 ──
        gr.Markdown("---")
        gr.Markdown("## 🆕 새 클립 생성")
        with gr.Row():
            with gr.Column():
                gen_prompt = gr.Textbox(label="생성 프롬프트", placeholder="예: a rocket launching into space at sunset", lines=2)
                gen_duration = gr.Slider(1, 20, value=5, step=1, label="길이 (초)")
                gen_backend = gr.Radio(
                    choices=["opencv"],
                    value="opencv", label="백엔드",
                )
                gen_btn = gr.Button("🆕 생성", variant="primary")
            with gr.Column():
                gen_status_md = gr.Markdown("")
                gen_preview_video = gr.Video(label="생성 미리보기")

        # ── Section E: 합성 ──
        gr.Markdown("---")
        gr.Markdown("## 🎬 영상 합성")
        with gr.Row():
            with gr.Column(scale=1):
                order_box = gr.Textbox(
                    label="배치 순서 (줄 순서 = 영상 순서, 직접 수정 가능)",
                    lines=8,
                    placeholder="클립 선택 시 자동 채워짐. 줄을 이동하면 순서가 바뀝니다.",
                )
            with gr.Column(scale=1):
                sort_radio = gr.Radio(
                    choices=["manual", "auto", "hybrid"], value="manual",
                    label="정렬 방식",
                    info="manual: 왼쪽 순서 그대로 / auto: 시각 유사도 자동 / hybrid: 첫·끝 고정",
                )
                assemble_btn = gr.Button("🎬 영상 합성", variant="primary", size="lg")
                reset_btn = gr.Button("🗑 클립 목록 초기화", variant="stop")
        assemble_result_md = gr.Markdown("")
        with gr.Row():
            result_video = gr.Video(label="합성 영상")
            assembly_plot = gr.Plot(label="합성 레이턴시")

        # ── Tab 2 이벤트 연결 ──

        # A. 검색 (누적 모드)
        search_btn.click(
            fn=on_search,
            inputs=[cur_query, cur_topk, ws_state],
            outputs=[ws_state, search_progress_md, assemble_result_md, clips_cbg, search_latency_plot, order_box],
        )

        # A-2. 초기화
        reset_btn.click(
            fn=on_reset_clips,
            inputs=[ws_state],
            outputs=[ws_state, clips_cbg, search_progress_md, assemble_result_md, search_latency_plot],
        )

        # B. 클립 선택 -> 드롭다운 + 순서 박스 동기화
        def _sync_selection(selected, state):
            dd_choices = [s for s in selected] if selected else []
            order_lines = []
            for s in (selected or []):
                clip = _find_clip(state, s)
                if clip:
                    tag = {"archive": "📁", "transformed": "🔄", "generated": "🆕"}.get(clip["source"], "")
                    order_lines.append(f'{tag} {clip["clip_id"]} — {clip["caption"][:40]}')
            return gr.update(choices=dd_choices, value=dd_choices[-1] if dd_choices else None), "\n".join(order_lines)

        clips_cbg.change(fn=_sync_selection, inputs=[clips_cbg, ws_state], outputs=[clip_detail_dd, order_box])

        # B. 클립 상세 보기
        clip_detail_dd.change(fn=on_clip_detail, inputs=[clip_detail_dd, ws_state], outputs=[clip_preview_video])

        # C. 변환 (비활성 — 2차년도 구현 예정)

        # D. 생성
        gen_btn.click(
            fn=on_generate,
            inputs=[gen_prompt, gen_duration, gen_backend, ws_state],
            outputs=[ws_state, gen_status_md, gen_preview_video, clips_cbg],
        )

        # B. 제거 (비활성 — 체크박스 해제로 대체)

        # E. 합성
        assemble_btn.click(
            fn=on_assemble,
            inputs=[order_box, clips_cbg, sort_radio, ws_state],
            outputs=[ws_state, assemble_result_md, result_video, assembly_plot],
        )

print("Gradio 앱 준비 완료.")
print("(share=True → 외부 접속 가능한 public URL이 생성됩니다)\n")
demo.launch(share=True)
